In [ ]:
import os
import requests
import pandas as pd
import json
from dotenv import load_dotenv

# Cargar las variables de entorno del archivo .env
load_dotenv("/home/jovyan/work/.env")

from sqlalchemy import create_engine

def cargar_system_prompt():
    ruta_prompt = '/home/jovyan/work/notebooks/prompt_sistema.md'
    try: 
        with open(ruta_prompt, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print("Error: no se encontró el archivo de prompt de sistema.")
        return ""

def text_to_sql(pregunta_usuario):
    url = os.getenv("OLLAMA_URL", "http://ollama:11434/api/generate")
    model = os.getenv("LLM_MODEL", "llama3.2")
    
    system_prompt = cargar_system_prompt()
    if not system_prompt:
        raise Exception("No se pudo cargar el prompt del sistema.")

    full_prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTraduce esta pregunta a SQL: {pregunta_usuario}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    
    payload = {
        "model": model,
        "prompt": full_prompt,
        "stream": True,
        "options": {
            "temperature": 0.0 # Creatividad cero para aumentar precisión
        }
    }
    
    # Para stream True
    response = requests.post(url, json=payload, stream=True)

    sql_acumulado = ""

    for line in response.iter_lines():
        if line:
            # Ollama manda JSONs separados por línea
            chunk = json.loads(line.decode('utf-8'))
            texto_pedazo = chunk.get('response', '')
            
            # Imprimimos el pedacito en la consola al instante sin saltar de línea
            print(texto_pedazo, end="", flush=True)
            
            sql_acumulado += texto_pedazo
            
    return sql_acumulado.strip()

    """
    # para stream False
    resultado = response.json()

    sql_generado = resultado['response'].strip()
    return sql_generado
    """

   
def ejecutar_consulta(sql):
    # SQLAlchemy
    host = os.getenv("DB_HOST")
    port = os.getenv("DB_PORT")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")
    
    conexion_url = f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}"
    
    try:
        engine = create_engine(conexion_url)
        
        with engine.connect() as conn:
            df = pd.read_sql(sql, conn)
        return df
    except Exception as e:
        # print(f"Error sintáctico en la base de datos: \n{e}\n\n")
        raise e


def preguntar_al_agente(pregunta):
    print(f"Pregunta del usuario: \n> {pregunta}\n")
    try:
        print(f"SQL Generado por el modelo:",end="\n", flush=True)
        sql = text_to_sql(pregunta)
        print("\n")
        
        print("Conectando a base de datos...")
        df_resultado = ejecutar_consulta(sql)

        print("Resultado:")
        display(df_resultado)
        
    except Exception as e:
        print(f"Ocurrió un error:\n{e}\n\n")

In [ ]:
print("Prueba rápida de IA")
print("-" * 60)

preguntar_al_agente("cuáles son los 10 libros más prestados de medicina?")
# preguntar_al_agente("Hola!")

In [ ]:
print("¡Agente BiblioIA Activado! Escribí 'salir' para terminar.")
print("-" * 60)

while True:
    pregunta = input("\nIngresá tu pregunta para la biblioteca: ")
    if pregunta.lower() in ['salir', 'chau','adios','adiós','exit', 'quit']:
        print("Chau")
        break
    if pregunta.strip() == "":
        continue
        
    # Llama a tu función principal
    preguntar_al_agente(pregunta)

In [ ]:
import requests

try:
    # Cambiá 'llama3.2' por el modelo exacto que bajaron si usaron otro
    res = requests.post("http://ollama:11434/api/generate", 
                        json={"model": "llama3.2", "prompt": "Hola, estás vivo?", "stream": False}, 
                        timeout=60)
    print("Respuesta de Ollama:", res.json()['response'])
except Exception as e:
    print("Error al conectar con Ollama:", e)